In [1]:
import numpy as np
import tensorflow as tf

2026-04-27 15:20:31.025150: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 15:20:31.033283: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-27 15:20:31.043647: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-27 15:20:31.043663: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-27 15:20:31.051119: I tensorflow/core/platform/cpu_feature_gua

In [ ]:
X = np.memmap("X_AllStepsR_f32_norm.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
# X = np.memmap("X_AllStepsR_f32_normMM.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
Y = np.load("Y_AllStepsR.npy")       # class 0 -> 150
XRef = np.load("XRef_AllStepsR_f32_norm.npy")
YRef = np.load('YRef_AllStepsR.npy') # class 233 -> 247
YRef -= 80                           # class 153 -> 167

In [ ]:
X.shape, XRef.shape

### V4. V1 architecture but triplet loss addition

In [ ]:
BATCH_SIZE = 32
VAL_SPLIT = 0.05

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True, flat=True):
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch]
            Y_ = Y[batch, 0:1]

            if flat:
                X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
            yield X_, Y_

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
A, B = next(gen_train)

In [3]:
import inceptionembed
model = inceptionembed.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

2026-04-27 15:20:41.375552: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-27 15:20:41.413708: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-27 15:20:41.413743: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-27 15:20:41.418798: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-27 15:20:41.418830: I external/local_xla/xla/stream_executor

In [4]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 3000)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, None, 32)  │     96,000 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, None,      │          0 │ input_layer[0][0] │
│ (MaxPooling1D)      │ 3000)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, None, 32)  │     41,984 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, None, 32)  │     20,480 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, None, 32)  │     10,240 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, None, 32)  │     96,000 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None, 128) │          0 │ conv1d_1[0][0],   │
│ (Concatenate)       │                   │            │ conv1d_2[0][0],   │
│                     │                   │            │ conv1d_3[0][0],   │
│                     │                   │            │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, None, 128) │        512 │ concatenate[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, None, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, None, 32)  │      4,096 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, None, 128) │          0 │ activation[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, None, 32)  │     41,984 │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, None, 32)  │     20,480 │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, None, 32)  │     10,240 │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, None, 32)  │      4,096 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, None, 128) │          0 │ conv1d_6[0][0],   │
│ (Concatenate)       │                   │            │ conv1d_7[0][0],   │
│                     │                   │            │ conv1d_8[0][0],   │
│                     │                   │            │ conv1d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None, 128) │        512 │ concatenate_1[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, None, 128) │          0 │ batch_normalizat

 Total params: 1,099,464 (4.19 MB)

 Trainable params: 1,097,416 (4.19 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [ ]:
def triplet_loss(a,p,n,margin=0.3):
    dap = tf.reduce_sum(tf.square(a-p), axis=1)
    dan = tf.reduce_sum(tf.square(a-n), axis=1)
    return tf.reduce_mean(tf.maximum(dap - dan + margin, 0.0))

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(), # CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc") # tf.keras.metrics.CategoricalAccuracy(name="acc")
    ]
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(id_train)//BATCH_SIZE//2, # Smaller "fake epochs" for faster feedback
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[checkpoint]
)

In [ ]:
model.save('last_model.keras')
# model.save_weights('best_model.weights.h5')
# model.load_weights('best_model.weights.h5')

In [ ]:
model = tf.keras.models.load_model('best_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model = tf.keras.models.load_model('last_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

### V3. Improve model architecture with 2D then 1D

In [ ]:
BATCH_SIZE = 32
VAL_SPLIT = 0.05

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True, flat=True):
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch]
            Y_ = Y[batch, 0:1]

            if flat:
                X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            else:
                X_ = np.expand_dims(X_, axis=-1)
            
            yield X_, Y_

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True, flat=False)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False, flat=False)

In [ ]:
A, B = next(gen_train)
A[0].shape, A[-1].shape, B[0].shape, B[-1].shape, B[0], B[-1]

In [ ]:
import inceptionviz
# model = inceptionviz.get_model(n_classes=200, input_shape=(None, 75, 40, 1), reduce=[tf.keras.layers.GlobalAveragePooling1D()])
model = inceptionviz.get_model(n_classes=200, input_shape=(None, 75, 40, 1), reduce=[tf.keras.layers.GlobalAveragePooling1D()], encoder_version=2)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(), # CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc") # tf.keras.metrics.CategoricalAccuracy(name="acc")
    ]
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(id_train)//BATCH_SIZE//2, # Smaller "fake epochs" for faster feedback
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[checkpoint]
)

In [ ]:
model.save('last_model.keras')
# model.save_weights('best_model.weights.h5')
# model.load_weights('best_model.weights.h5')

In [ ]:
model = tf.keras.models.load_model('best_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model = tf.keras.models.load_model('last_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model.summary()

### V2. Try to train a classifier for X/Y and XRef/YRef

In [ ]:
BATCH_SIZE = 32
VAL_SPLIT = 0.05

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True, flat=True, ref=False, XR=None, YR=None):
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch].copy()
            Y_ = Y[batch, 0:1].copy()

            if ref:
                r = np.random.choice(len(XRef))
                X_[-1] = XR[r]
                Y_[-1] = YR[r//2, 0:1] # je devrais pas utiliser ça mais plutôt faire une data prep plus propre non ?

            if flat:
                X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
            yield X_, Y_

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True, ref=True, XR=XRef, YR=YRef)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
A, B = next(gen_train)
A[0].shape, A[-1].shape, B[0].shape, B[-1].shape, B[0], B[-1]

In [ ]:
import inception
model = inception.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(), # CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc") # tf.keras.metrics.CategoricalAccuracy(name="acc")
    ]
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(id_train)//BATCH_SIZE//2, # Smaller "fake epochs" for faster feedback
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[checkpoint]
)

In [ ]:
model.save('last_model.keras')
# model.save_weights('best_model.weights.h5')
# model.load_weights('best_model.weights.h5')

In [ ]:
model = tf.keras.models.load_model('best_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model = tf.keras.models.load_model('last_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model.summary()

### V1. Try to train a classifier for X/Y

In [ ]:
BATCH_SIZE = 32
# VAL_SPLIT = 0.05
VAL_SPLIT = 0.005 # reduce val split for 002 as we want to keep last model anyway

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True, flat=True):
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch]
            Y_ = Y[batch, 0:1]

            if flat:
                X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
            yield X_, Y_

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
A, B = next(gen_train)

In [ ]:
import inception
model = inception.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(), # CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc") # tf.keras.metrics.CategoricalAccuracy(name="acc")
    ]
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

class Every100Epochs(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 100 == 0:
            self.model.save(f'checkpoint_epoch_{epoch+1}.keras')
            print(f"\nSaved checkpoint at epoch {epoch+1}")

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(id_train)//BATCH_SIZE//2, # Smaller "fake epochs" for faster feedback
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=1000,
    callbacks=[checkpoint, Every100Epochs()]
)

In [ ]:
model.save('last_model.keras')
# model.save_weights('best_model.weights.h5')
# model.load_weights('best_model.weights.h5')

In [ ]:
model = tf.keras.models.load_model('best_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model = tf.keras.models.load_model('last_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)